# step00e - Surface soak: near-surface data quality, soak vs no-soak

**Question (Reez):** do casts with a surface soak (P45_06) have cleaner near-surface data than casts
where the soak was skipped or shortened (P45_07)? This quantifies it on the **same variable** across
the whole fleet of both cruises, so the comparison is fair.

**What it measures,** on the sectioned downcast, in the top `TOP_DB` decibars (the "near surface"):
- **Loop-edit flag %** - fraction of near-surface scans flagged (heave / slow or reversed motion).
- **Salinity spiking** - high-frequency wiggle (std of salinity minus its running median); the direct
  consequence of the conductivity cell and sensors not yet being equilibrated.
- **Temperature spiking** - same idea for temperature.
Each is also computed **below** the near-surface band (deep) as a within-cast baseline.

**Why a soak should help (mechanism):** a surface soak lets the pump prime, the conductivity cell and
plumbing flush, and the thermistor and oxygen membrane equilibrate before the descent. Skipping it
leaves the top of the cast with sensors still settling. This is standard Sea-Bird / GO-SHIP practice.

**Honest limitations - read before concluding:**
- This is **observational, cross-cruise**. P45_06 and P45_07 also differ in season, sea state and crew,
  so a difference is consistent with the soak helping, it does not prove soak alone caused it.
- Near-surface flags are partly **ship heave** (winch/swell), which a soak does not fix; steady descent
  does. So this measures near-surface quality, which soak and steady deployment both drive.
- The clean proof would be **alternating soak / no-soak casts within one cruise**, same conditions.
- Compared on the **same variable** (salinity, temperature) for both cruises, never oxygen-vs-temperature.

## 1. Setup

In [ ]:
from __future__ import annotations
import re, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")                      # static, reliable PNGs
import matplotlib.pyplot as plt
from matplotlib import rcParams
try:
    from IPython.display import display, Image
except Exception:
    def display(obj): print(obj)
    Image = None

# --------------------------------------------------------------- paths / config
CTD_ROOT    = Path(r"C:\Projects\ctd_pipeline")
SBE_BIN_DIR = Path(r"C:\Program Files (x86)\Sea-Bird\SBEDataProcessing-Win32")
PSA_DIR     = CTD_ROOT / "psa"
WORK        = CTD_ROOT / "_sbe_work" / "soak"; WORK.mkdir(parents=True, exist_ok=True)
OUT_DIR     = CTD_ROOT / "_audit" / "soak_comparison"; OUT_DIR.mkdir(parents=True, exist_ok=True)

# Which cruises used a soak vs not (edit to your cruises)
SOAK_CRUISES   = ["P45_06"]     # surface soak used
NOSOAK_CRUISES = ["P45_07"]     # soak skipped / shortened

PSA = {"filter": "05_filter.psa", "align": "02_alignctd.psa",
       "celltm": "04_celltm.psa", "loopedit": "06a_loopedit_percentmean.psa"}

TOP_DB = 10.0        # "near surface" band (decibars); below this = deep baseline
LAT    = 5.0
C_STD  = 4.2914

# ctd_lib (validated PSS-78 salinity + potential temperature); inline fallback if not importable
import sys; sys.path.insert(0, str(CTD_ROOT))
try:
    from ctd_lib import practical_salinity, potential_temperature
    _HAVE_LIB = True
except Exception as e:
    _HAVE_LIB = False
    print("ctd_lib not importable (", e, ") - using inline PSS-78 fallback.")

rcParams.update({"figure.dpi": 120, "savefig.dpi": 200, "font.size": 9, "axes.titlesize": 10,
                 "axes.labelsize": 9, "axes.linewidth": 0.8, "xtick.direction": "in",
                 "ytick.direction": "in", "xtick.labelsize": 8, "ytick.labelsize": 8,
                 "legend.fontsize": 8, "axes.grid": False, "font.family": "DejaVu Sans"})
C_SOAK, C_NOSOAK = "#2E6FB0", "#D62728"

def new_static_fig(nrows=1, ncols=1, **kw):
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    fig = Figure(**kw); FigureCanvasAgg(fig)
    return fig, fig.subplots(nrows, ncols, squeeze=False)

def finish(fig, out):
    try: fig.savefig(out); ok = True
    except Exception as e: print("savefig failed:", e); ok = False
    if ok and Image is not None:
        try: display(Image(filename=str(out)))
        except Exception: display(fig)
    return out if ok else None

def _despine(ax):
    for s in ("top", "right"): ax.spines[s].set_visible(False)

print("soak cruises:", SOAK_CRUISES, "| no-soak cruises:", NOSOAK_CRUISES, "| near-surface band: top", TOP_DB, "db")

## 2. Readers, salinity, spiking (identical to step00d)

In [ ]:
def parse_cnv_raw(path):
    """Parse a .cnv keeping ALL scans (incl. flagged). Returns a numeric DataFrame or None."""
    if path is None: return None
    text = Path(path).read_text(encoding="latin-1", errors="replace").splitlines()
    names, ds = {}, None
    nr = re.compile(r"#\s*name\s+(\d+)\s*=\s*([^:]+?)\s*[:=]", re.I)
    for i, ln in enumerate(text):
        s = ln.strip()
        if s.startswith("#") or s.startswith("*"):
            m = nr.search(ln)
            if m: names[int(m.group(1))] = m.group(2).strip()
            if s.upper().startswith("*END*"): ds = i + 1; break
    if ds is None: return None
    cols = [names[k] for k in sorted(names)]
    rows = [l.split()[:len(cols)] for l in text[ds:] if l.strip() and len(l.split()) >= len(cols)]
    df = pd.DataFrame(rows, columns=cols).apply(pd.to_numeric, errors="coerce").reset_index(drop=True)
    # Sea-Bird bad-flag sentinel -> NaN so it never plots or enters a statistic. The sentinel is
    # -9.990e-29 (a TINY negative number, not a large one), which Align CTD etc. write into the record
    # ends they shift out of. Bracket it tightly; real T/C/P/S/O2 values never fall in this range.
    bad = (df.le(-8e-29) & df.ge(-1.2e-28)) | (df.abs() > 1e30)
    # IMPORTANT: the loop-edit "flag" column legitimately uses -9.990e-29 to mark a FLAGGED scan
    # (0 = good). Do not NaN it, or the red flagged points and the flag % would vanish.
    for fc in ("flag",):
        if fc in bad.columns:
            bad[fc] = False
    return df.mask(bad)


def read_cnv_full(path):
    """(header_lines, cols, rows_raw) so a sectioned .cnv can be re-written with its header."""
    txt = Path(path).read_text(encoding="latin-1", errors="replace").splitlines()
    names, ds = {}, None
    nr = re.compile(r"#\s*name\s+(\d+)\s*=\s*([^:]+?)\s*[:=]", re.I)
    for i, l in enumerate(txt):
        s = l.strip()
        if s.startswith("#") or s.startswith("*"):
            m = nr.search(l)
            if m: names[int(m.group(1))] = m.group(2).strip()
            if s.upper().startswith("*END*"): ds = i + 1; break
    cols = [names[k] for k in sorted(names)]
    rows_raw = [l for l in txt[ds:] if l.strip()]
    return txt[:ds], cols, rows_raw


def pick(df, *cands):
    if df is None: return None
    for c in cands:
        if c in df.columns: return c
    for c in df.columns:
        if any(x.lower() in c.lower() for x in cands): return c
    return None


def col_P(df): return pick(df, "prdM", "prDM", "prM", "prSM", "pressure")
def col_T(df): return pick(df, "tv290C", "t090C", "tv290", "t190C", "temperature")
def col_time(df): return pick(df, "timeS", "timeJ", "scan")


def cond_Sm(df):
    """Conductivity column as a numpy array in S/m (converts mS/cm -> S/m)."""
    c = pick(df, "c0S/m", "cond0S/m", "c_S/m", "c0mS/cm", "c0ms/cm", "cond0mS/cm", "conductivity")
    if c is None: return None, None
    x = pd.to_numeric(df[c], errors="coerce").to_numpy(float)
    if "ms/cm" in c.lower():
        x = x * 0.1
    return x, c


def flag_mask(df, flag_col="flag"):
    """Boolean mask of loop-edit flagged scans (flag != 0), or None if no flag column."""
    if df is None or flag_col not in df.columns:
        return None
    return pd.to_numeric(df[flag_col], errors="coerce").fillna(0) != 0


def split_down_up(df, pcol):
    """Split a stage DataFrame into (downcast, upcast) at the deepest scan."""
    p = pd.to_numeric(df[pcol], errors="coerce")
    imax = int(p.idxmax()) if p.notna().any() else len(df) - 1
    return df.iloc[:imax + 1], df.iloc[imax:]


def num(df, col):
    return pd.to_numeric(df[col], errors="coerce").to_numpy(float) if (df is not None and col in df.columns) else None


# ---- salinity + density -----------------------------------------------------
if not _HAVE_LIB:
    def _sal78(R, T68, P):
        a=[0.0080,-0.1692,25.3851,14.0941,-7.0261,2.7081]; b=[0.0005,-0.0056,-0.0066,-0.0375,0.0636,-0.0144]
        c=[0.6766097,2.00564e-2,1.104259e-4,-6.9698e-7,1.0031e-9]; d=[3.426e-2,4.464e-4,4.215e-1,-3.107e-3]
        e=[2.070e-5,-6.370e-10,3.989e-15]; R=np.asarray(R,float)
        Rp=1+(P*(e[0]+e[1]*P+e[2]*P*P))/(1+d[0]*T68+d[1]*T68*T68+(d[2]+d[3]*T68)*R)
        rt=c[0]+c[1]*T68+c[2]*T68**2+c[3]*T68**3+c[4]*T68**4; Rt=np.maximum(R/(Rp*rt),1e-12); s=np.sqrt(Rt)
        dS=(T68-15)/(1+0.0162*(T68-15))*(b[0]+b[1]*s+b[2]*Rt+b[3]*Rt*s+b[4]*Rt**2+b[5]*Rt**2*s)
        return a[0]+a[1]*s+a[2]*Rt+a[3]*Rt*s+a[4]*Rt**2+a[5]*Rt**2*s+dS
    def practical_salinity(C_Sm, T90, P):
        return _sal78(np.asarray(C_Sm,float)/C_STD, np.asarray(T90,float)*1.00024, P)
    def _adtg(S,T,P):
        a0,a1,a2,a3=3.5803e-5,8.5258e-6,-6.836e-8,6.6228e-10; b0,b1=1.8932e-6,-4.2393e-8
        c0,c1,c2,c3=1.8741e-8,-6.7795e-10,8.733e-12,-5.4481e-14; d0,d1=-1.1351e-10,2.7759e-12
        e0,e1,e2=-4.6206e-13,1.8676e-14,-2.1687e-16
        return (a0+(a1+(a2+a3*T)*T)*T+(b0+b1*T)*(S-35)+((c0+(c1+(c2+c3*T)*T)*T)+(d0+d1*T)*(S-35))*P+(e0+(e1+e2*T)*T)*P*P)
    def potential_temperature(S,T90,P,PR=0.0):
        S=np.asarray(S,float); T=np.asarray(T90,float)*1.00024; P=np.asarray(P,float); H=PR-P
        XK=H*_adtg(S,T,P); T=T+0.5*XK; Q=XK; P2=P+0.5*H
        XK=H*_adtg(S,T,P2); T=T+0.29289322*(XK-Q); Q=0.58578644*XK+0.121320344*Q
        XK=H*_adtg(S,T,P2); T=T+1.707106781*(XK-Q); Q=3.414213562*XK-4.121320344*Q
        P3=P2+0.5*H; XK=H*_adtg(S,T,P3); T=T+(XK-2.0*Q)/6.0
        return T/1.00024


def _rho_p0(S, T68):
    """EOS-80 density at one atmosphere (p=0), kg/m3. Checked: rho(35,5,0)=1027.675."""
    T = T68
    rw = (999.842594 + 6.793952e-2*T - 9.095290e-3*T**2 + 1.001685e-4*T**3
          - 1.120083e-6*T**4 + 6.536332e-9*T**5)
    A = 8.24493e-1 - 4.0899e-3*T + 7.6438e-5*T**2 - 8.2467e-7*T**3 + 5.3875e-9*T**4
    B = -5.72466e-3 + 1.0227e-4*T - 1.6546e-6*T**2
    Cc = 4.8314e-4
    return rw + A*S + B*S**1.5 + Cc*S**2


def sigma_theta(C_Sm, T90, P):
    """sigma-theta (kg/m3) from conductivity[S/m], T90, P: potential temp to 0 dbar, EOS-80 at p=0."""
    S = practical_salinity(C_Sm, T90, P)
    th90 = potential_temperature(S, T90, P)
    return _rho_p0(S, np.asarray(th90, float) * 1.00024) - 1000.0


def salinity_of(df):
    """Practical salinity from a stage DataFrame (C in S/m, T90, P), or None if columns missing."""
    cS, _ = cond_Sm(df); p = col_P(df); t = col_T(df)
    if cS is None or p is None or t is None: return None
    return practical_salinity(cS, num(df, t), num(df, p))


# ---- metric helpers ---------------------------------------------------------
def value_delta(xb, xa, pressure=None, res=None):
    """Per-variable change: after - before on the common finite scans."""
    n = min(len(xb), len(xa)); xb = np.asarray(xb[:n], float); xa = np.asarray(xa[:n], float)
    d = xa - xb; m = np.isfinite(d) & np.isfinite(xb)
    if m.sum() < 3:
        return dict(n=0, rms=np.nan, nrms=np.nan, mean=np.nan, max_abs=np.nan, p_at_max=np.nan, pct_moved=np.nan)
    dd = d[m]; base_sd = np.nanstd(xb[m])
    out = dict(n=int(m.sum()), rms=float(np.sqrt(np.mean(dd**2))), mean=float(np.mean(dd)),
               max_abs=float(np.max(np.abs(dd))),
               nrms=float(np.sqrt(np.mean(dd**2)) / base_sd) if base_sd > 0 else np.nan,
               pct_moved=float(100*np.mean(np.abs(dd) > res)) if res else np.nan)
    if pressure is not None:
        pr = np.asarray(pressure[:n], float)[m]
        out["p_at_max"] = float(pr[int(np.argmax(np.abs(dd)))]) if pr.size else np.nan
    else:
        out["p_at_max"] = np.nan
    return out


def dxdt_std(x):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    return float(np.nanstd(np.gradient(x))) if x.size > 3 else np.nan


def spike_index(S, w=7):
    """High-frequency salinity wiggle: std of (S - running median). Alignment should lower it."""
    s = pd.Series(np.asarray(S, float))
    resid = s - s.rolling(w, center=True, min_periods=1).median()
    return float(np.nanstd(resid.to_numpy()))


def thermocline_mask(T, P, frac=0.25):
    """Scans in the strongest-gradient band (top `frac` of |dT/dP|)."""
    T = np.asarray(T, float); P = np.asarray(P, float)
    g = np.abs(np.gradient(T, P))
    g = np.where(np.isfinite(g), g, 0.0)
    if not np.any(g > 0): return np.zeros(len(T), bool)
    thr = np.nanquantile(g[g > 0], 1 - frac)
    return g >= thr


def density_inversions(C_Sm, T90, P, thr=0.005):
    """On the downcast (increasing P), count sigma-theta decreases > thr (static instability)."""
    P = np.asarray(P, float); imax = int(np.nanargmax(P)) if np.isfinite(P).any() else 0
    sl = slice(0, imax + 1)
    st = sigma_theta(np.asarray(C_Sm)[sl], np.asarray(T90)[sl], P[sl])
    pr = P[sl]; ok = np.isfinite(st) & np.isfinite(pr)
    st, pr = st[ok], pr[ok]
    if st.size < 5: return 0, np.nan
    order = np.argsort(pr); dd = np.diff(st[order])
    neg = dd[dd < -thr]
    return int(neg.size), (float(-neg.min()) if neg.size else 0.0)

## 3. Process each cast to Loop Edit (same chain as step01/step00d, stopping at Loop Edit)

In [ ]:
def run_sbe(exe, psa_name, in_cnv, out_stem):
    psa = PSA_DIR / psa_name
    if not psa.exists():
        print(f"    MISSING PSA: {psa}  <-- this .psa does not exist (maybe only a .bak). {exe} skipped,"
              f" so this and all later stages are empty.")
        return None
    if in_cnv is None:                                   # upstream stage failed; do not chain onward
        return None
    cmd = [str(SBE_BIN_DIR / exe), f"/p{psa}", f"/i{in_cnv}", f"/o{WORK}", f"/f{out_stem}", "/s"]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        print(f"    WARNING {exe} rc={res.returncode}: {(res.stderr or '')[:160]}")
    # find the output regardless of any NameAppend inside the PSA (e.g. a leftover '_1s')
    cands = sorted(WORK.glob(f"{out_stem}*.cnv"), key=lambda q: q.stat().st_mtime, reverse=True)
    return cands[0] if cands else None


def check_psas():
    """Pre-flight: list any required .psa that is missing (the usual cause of empty later stages)."""
    missing = [n for n in PSA.values() if not (PSA_DIR / n).exists()]
    if missing:
        print("MISSING active PSA files (create these - a .bak backup is not loaded):")
        for n in missing:
            print("   ", PSA_DIR / n)
    else:
        print("PSA pre-flight OK: all required .psa present.")
    return not missing


# Section step: IDENTICAL to step01 (section_cnv / _descent_onset) so the audit reproduces the
# production pipeline EXACTLY - same descent onset (fs-scan window + degeneracy guard) and a
# byte-for-byte sectioned .cnv (CRLF preserved, only '# nvalues' updated). Do not change this, or the
# audit would diverge from step01 at Section and every stage after it (Loop Edit, Derive, Bin Avg).
SECTION_SOAK_MIN_DROP_DB = 0.10   # min pressure increase over ~1 s to count as descending (step01)
SECTION_DEFAULT_FS = 24.0         # fallback sample rate if the header has no interval (step01)


def _cnv_pressure_column(header_lines):
    name_re = re.compile(r"#\s*name\s+(\d+)\s*=\s*([^:]+?)\s*[:=]", re.I)
    best = None
    for line in header_lines:
        m = name_re.search(line)
        if not m: continue
        idx = int(m.group(1)); label = m.group(2).strip().lower()
        if label in ("prdm", "prm", "prsm") or label.startswith("prdm"): return idx
        if best is None and "pr" in label and "spar" not in label: best = idx
    return best


def _cnv_sample_rate(header_lines):
    for line in header_lines:
        low = line.lower()
        if "interval" in low and "seconds" in low:
            m = re.search(r"seconds\s*:\s*([0-9.eE+-]+)", line)
            if m:
                try:
                    dt = float(m.group(1))
                    if dt > 0: return 1.0 / dt
                except ValueError: pass
    return SECTION_DEFAULT_FS


def _descent_onset(pressure, fs):
    """step01's descent onset: walk back from the deepest scan over a ~1 s (fs-scan) window; fall
    back to the surface crossing if that is degenerate (too short / <10 db)."""
    p = np.asarray(pressure, float)
    if p.size == 0 or np.isnan(p).all(): return 0
    imax = int(np.nanargmax(p))
    win = max(int(round(fs)), 1)
    if imax < win + 1: return 0
    seg = pd.Series(p[:imax + 1]).rolling(win, center=True, min_periods=1).median().to_numpy()
    fwd = np.full(len(seg), np.nan); fwd[:len(seg) - win] = seg[win:] - seg[:len(seg) - win]
    descending = np.where(np.isnan(fwd), True, fwd > SECTION_SOAK_MIN_DROP_DB)
    i = imax
    while i > 0 and descending[i - 1]: i -= 1
    start = i
    span_db = (p[imax] - p[start]) if imax > start else 0.0
    if (imax - start) < int(5 * fs) or span_db < 10.0:
        below = np.where(p[:imax + 1] < 8.0)[0]
        start = int(below[-1]) if below.size else 0
    return start


def section_to_cnv(src_cnv, out_cnv):
    """Byte-for-byte sectioned .cnv (step01's section_cnv): keep [onset:end], CRLF preserved, only
    '# nvalues' updated. Returns the output path."""
    src = Path(src_cnv); out = Path(out_cnv)
    lines = src.read_bytes().splitlines(keepends=True)
    end_idx = None
    for k, line in enumerate(lines):
        if line.strip().upper().startswith(b"*END*"): end_idx = k; break
    if end_idx is None: raise ValueError(f"No *END* in {src.name}")
    header = lines[:end_idx + 1]
    data_rows = [ln for ln in lines[end_idx + 1:] if ln.strip() != b""]
    if not data_rows: raise ValueError(f"{src.name} has no data rows")
    header_text = [ln.decode("latin-1", "replace").rstrip("\r\n") for ln in header]
    pcol = _cnv_pressure_column(header_text)
    if pcol is None: raise ValueError(f"no pressure column in {src.name}")
    fs = _cnv_sample_rate(header_text)
    pressure = np.full(len(data_rows), np.nan)
    for k, ln in enumerate(data_rows):
        parts = ln.decode("latin-1", "replace").split()
        if pcol < len(parts):
            try: pressure[k] = float(parts[pcol])
            except ValueError: pressure[k] = np.nan
    start = _descent_onset(pressure, fs)
    kept = data_rows[start:]
    nval_re = re.compile(rb"(#\s*nvalues\s*=\s*)(\d+)", re.I)
    new_header = [nval_re.sub(lambda mm: mm.group(1) + str(len(kept)).encode("ascii"), ln)
                  if nval_re.search(ln) else ln for ln in header]
    with open(out, "wb") as fh:
        fh.writelines(new_header); fh.writelines(kept)
    return out


def casts_in(cruise):
    d = CTD_ROOT / "cruises" / cruise / "L1" / "CTD"
    return sorted(p.name[:-len(".cnv")] for p in d.glob("*.cnv")) if d.exists() else []


P_SUBMERGE_DB = 0.5   # pressure (db) at/above which the package counts as in the water

def _pressure_fs(cnv_path):
    """(pressure array, fs) from a .cnv, using the same header helpers as Section."""
    lines = Path(cnv_path).read_bytes().splitlines(keepends=True)
    end_idx = next((k for k, ln in enumerate(lines) if ln.strip().upper().startswith(b"*END*")), None)
    if end_idx is None: return None, SECTION_DEFAULT_FS
    header = [l.decode("latin-1", "replace").rstrip("\r\n") for l in lines[:end_idx + 1]]
    pcol = _cnv_pressure_column(header); fs = _cnv_sample_rate(header)
    if pcol is None: return None, fs
    rows = [l for l in lines[end_idx + 1:] if l.strip() != b""]
    p = np.full(len(rows), np.nan)
    for k, ln in enumerate(rows):
        parts = ln.decode("latin-1", "replace").split()
        if pcol < len(parts):
            try: p[k] = float(parts[pcol])
            except ValueError: pass
    return p, fs

def soak_seconds(cnv_path):
    """Surface soak duration (s) from a PRE-section cnv (the Cell TM output still has the soak): time
    between first submersion (P >= P_SUBMERGE_DB) and the descent onset Section uses. Returns
    (soak_s, n_soak_scans). The same _descent_onset() as Section defines the boundary, so 'soak' here
    is exactly the in-water time Section later trims off the top of the cast."""
    try:
        p, fs = _pressure_fs(cnv_path)
    except Exception:
        return np.nan, 0
    if p is None or not np.isfinite(p).any() or not fs or fs <= 0:
        return np.nan, 0
    onset = _descent_onset(p, fs)
    sub = np.where(p >= P_SUBMERGE_DB)[0]
    first_in = int(sub[0]) if sub.size else 0
    n_soak = max(onset - first_in, 0)
    return (n_soak / fs, int(n_soak))


def process_cast(cruise, cast):
    """Filter -> Align -> Cell TM -> Section -> Loop Edit; return the sectioned cast (with flags)."""
    L1 = CTD_ROOT / "cruises" / cruise / "L1" / "CTD" / f"{cast}.cnv"
    if not L1.exists():
        return None
    f  = run_sbe("FilterW.exe",   PSA["filter"],   L1, f"{cast}_s1")
    a  = run_sbe("AlignCTDW.exe", PSA["align"],    f,  f"{cast}_s2")
    ct = run_sbe("CellTMW.exe",   PSA["celltm"],   a,  f"{cast}_s3")
    sk_s, sk_n = soak_seconds(ct) if ct else (np.nan, 0)      # soak measured BEFORE Section trims it
    sec = section_to_cnv(ct, WORK / f"{cast}_s4.cnv") if ct else None
    le = run_sbe("LoopEditW.exe", PSA["loopedit"], sec, f"{cast}_s5") if sec else None
    df = parse_cnv_raw(le) if le else None
    if df is not None:
        df.attrs["soak_s"] = sk_s; df.attrs["n_soak"] = sk_n   # carried to soak_metrics via .attrs
    return df

## 4. Near-surface soak-quality metrics

In [ ]:
def soak_metrics(df):
    """On the sectioned DOWNCAST: loop-edit flag %, salinity and temperature spiking, in the top
    TOP_DB db (near surface) and below it (deep baseline)."""
    if df is None: return None
    pcol = col_P(df)
    if pcol is None: return None
    p = num(df, pcol)
    if p is None or not np.isfinite(p).any(): return None
    imax = int(np.nanargmax(p))
    dn = df.iloc[:imax + 1]                      # downcast only
    pdn = num(dn, pcol)
    top = pdn <= TOP_DB
    deep = pdn > TOP_DB
    out = {"n_down": int(len(dn)), "n_top": int(np.nansum(top)),
           "soak_s": float(df.attrs.get("soak_s", np.nan))}   # surface soak duration (s)
    fm = flag_mask(dn)
    if fm is not None:
        fa = fm.to_numpy()
        out["flag_pct_top"]  = float(100 * np.mean(fa[top]))  if top.sum()  else np.nan
        out["flag_pct_deep"] = float(100 * np.mean(fa[deep])) if deep.sum() else np.nan
    S = salinity_of(dn); T = num(dn, col_T(dn))
    if S is not None:
        out["sal_spike_top"]  = spike_index(np.asarray(S)[top])  if top.sum()  > 5 else np.nan
        out["sal_spike_deep"] = spike_index(np.asarray(S)[deep]) if deep.sum() > 5 else np.nan
    if T is not None:
        out["temp_spike_top"]  = spike_index(T[top])  if top.sum()  > 5 else np.nan
        out["temp_spike_deep"] = spike_index(T[deep]) if deep.sum() > 5 else np.nan
    return out

## 5. Run both cruise groups

In [ ]:
rows = []
for group, cruises in (("soak", SOAK_CRUISES), ("no soak", NOSOAK_CRUISES)):
    for cruise in cruises:
        cl = casts_in(cruise)
        if not cl:
            print(f"  {cruise} ({group}): no L1 cnv found - run step01 for this cruise first."); continue
        print(f"{cruise} ({group}): {len(cl)} casts")
        for cast in cl:
            try:
                m = soak_metrics(process_cast(cruise, cast))
                if m:
                    m.update(group=group, cruise=cruise, cast=cast); rows.append(m)
            except Exception as e:
                print(f"    {cast}: {e}")

soak = pd.DataFrame(rows)
if not soak.empty:
    soak.to_csv(OUT_DIR / "soak_quality_by_cast.csv", index=False, encoding="utf-8-sig")
    print(f"\n{len(soak)} casts  ->  {OUT_DIR / 'soak_quality_by_cast.csv'}")
else:
    print("No casts processed; check cruise names and that step01 produced L1 cnvs.")

## 6. Comparison figure + summary

In [ ]:
def _strip(ax, metric, title, ylabel):
    rng = np.random.default_rng(1)
    for i, g in enumerate(("soak", "no soak")):
        v = pd.to_numeric(soak[soak["group"] == g][metric], errors="coerce").dropna().to_numpy()
        if not len(v): continue
        col = C_SOAK if g == "soak" else C_NOSOAK
        ax.scatter(i + rng.uniform(-0.12, 0.12, len(v)), v, s=20, color=col, alpha=0.7, edgecolors="none")
        med = float(np.median(v))
        ax.plot([i - 0.26, i + 0.26], [med, med], color="k", lw=2)
        ax.annotate(f"median {med:.3g}\n(n={len(v)})", (i, med), textcoords="offset points",
                    xytext=(0, 9), ha="center", fontsize=7.5)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["soak", "no soak"]); ax.set_xlim(-0.5, 1.5)
    ax.set_title(title, fontsize=9.5); ax.set_ylabel(ylabel); _despine(ax)

if not soak.empty:
    fig, axes = new_static_fig(1, 3, figsize=(11.5, 4.6))
    axes = axes.ravel()
    _strip(axes[0], "flag_pct_top",  "Loop-edit flags, top %g db" % TOP_DB, "% of downcast scans")
    _strip(axes[1], "sal_spike_top", "Salinity spiking, top %g db" % TOP_DB, "std of S - running median (PSU)")
    _strip(axes[2], "temp_spike_top","Temperature spiking, top %g db" % TOP_DB, "std of T - running median (degC)")
    fig.suptitle("Near-surface data quality: soak vs no-soak  (each point = one cast)", fontsize=12, y=0.99)
    fig.text(0.01, 0.005, "Observational, cross-cruise: consistent with the soak helping, not proof. "
             "Near-surface flags are partly ship heave. Same variable used for both cruises.",
             fontsize=7, color="#666666", va="bottom")
    fig.tight_layout(rect=[0, 0.04, 1, 0.95])
    finish(fig, OUT_DIR / "soak_comparison.png")

    # summary table: median per group, near-surface vs deep, and the top/deep contrast
    def med(group, col):
        v = pd.to_numeric(soak[soak["group"] == group][col], errors="coerce").dropna()
        return float(v.median()) if len(v) else np.nan
    print("\nMedian by group (near-surface top %g db, and deep baseline below it):" % TOP_DB)
    hdr = f"{'metric':<26}{'soak':>12}{'no soak':>12}"
    print(hdr); print("-" * len(hdr))
    for col, lab in (("flag_pct_top", "flag % (top)"), ("flag_pct_deep", "flag % (deep)"),
                     ("sal_spike_top", "sal spike (top)"), ("sal_spike_deep", "sal spike (deep)"),
                     ("temp_spike_top", "temp spike (top)"), ("temp_spike_deep", "temp spike (deep)")):
        print(f"{lab:<26}{med('soak', col):>12.4g}{med('no soak', col):>12.4g}")
    # within-cast contrast: how much worse the near-surface is than deep (ratio), per group
    print("\nNear-surface vs deep (top/deep ratio of salinity spiking; >1 = surface noisier):")
    for g in ("soak", "no soak"):
        st, sd = med(g, "sal_spike_top"), med(g, "sal_spike_deep")
        print(f"  {g:<9} {st/sd:.2f}" if (sd and np.isfinite(sd) and sd > 0) else f"  {g:<9} n/a")
    print("\nReading it: lower near-surface flag % and spiking = cleaner near-surface data. If the soak"
          " group is systematically lower across casts, that supports keeping the soak. See the caveats"
          " in the intro before stating it as cause.")

## 7. Significance tests (same variable; between groups and within cast)

Two non-parametric tests, so no normality assumption, both on the **same variable** across cruises:

- **Between groups (Mann-Whitney U):** is a near-surface metric different between the soak and no-soak
  casts? Reported with a rank-biserial effect size (how large, not just whether p < 0.05).
- **Within cast (Wilcoxon signed-rank):** is each cast's near-surface noisier than *its own* deep water?
  This is **internally controlled**: season, crew and sea state cancel within a single cast, so it is the
  piece of evidence least exposed to the cross-cruise confound.

Both fall back to a tie- and continuity-corrected normal approximation if `scipy` is not installed, so a
p-value always prints. For the exact tests, `pip install scipy` on the processing machine.

*(A dose-response test, soak seconds vs spiking, needs a per-cast soak time; that lives in
`soak_quality_analysis.ipynb`, which measures soak duration per cast. step00e groups by cruise instead.)*

In [ ]:
from math import erf, sqrt as _sqrt

def mannwhitney(a, b):
    """Two-sided Mann-Whitney U (a = soak, b = no-soak) with a rank-biserial effect size.
    scipy if available, else a tie- and continuity-corrected normal approximation so a p-value always
    prints. rank-biserial rb > 0 means group a (soak) tends to LARGER values (i.e. noisier)."""
    a = np.asarray(a, float); a = a[np.isfinite(a)]
    b = np.asarray(b, float); b = b[np.isfinite(b)]
    na, nb = len(a), len(b)
    if na < 2 or nb < 2:
        return dict(U=np.nan, p=np.nan, rb=np.nan, na=na, nb=nb, method="too few")
    try:
        from scipy.stats import mannwhitneyu
        U, p = mannwhitneyu(a, b, alternative="two-sided"); method = "scipy"
    except Exception:
        allv = np.concatenate([a, b])
        _, inv, cnt = np.unique(allv, return_inverse=True, return_counts=True)
        order = allv.argsort(); r = np.empty(len(allv)); r[order] = np.arange(1, len(allv) + 1)
        sums = np.zeros(len(cnt)); np.add.at(sums, inv, r); ranks = (sums / cnt)[inv]
        U = ranks[:na].sum() - na * (na + 1) / 2.0            # U for group a (soak)
        n = na + nb; mu = na * nb / 2.0
        tie = float((cnt ** 3 - cnt).sum())
        var = na * nb / 12.0 * ((n + 1) - tie / (n * (n - 1)))
        if var <= 0:
            p = 1.0
        else:
            z = (abs(U - mu) - 0.5) / _sqrt(var)              # continuity correction
            p = min(max(2 * (1 - 0.5 * (1 + erf(z / _sqrt(2)))), 0.0), 1.0)
        method = "normal-approx (pip install scipy for the exact test)"
    rb = 1.0 - 2.0 * U / (na * nb)
    return dict(U=float(U), p=float(p), rb=float(rb), na=na, nb=nb, method=method)


def wilcoxon_paired(top, deep):
    """Within-cast top-vs-deep paired test: is the near-surface noisier than the same cast's deep
    water? scipy if present, else a signed-rank normal approximation. med_diff > 0 = surface noisier."""
    top = np.asarray(top, float); deep = np.asarray(deep, float)
    m = np.isfinite(top) & np.isfinite(deep); d = top[m] - deep[m]
    nz = d[d != 0]; n = len(nz)
    base = dict(n=int(n), med_diff=float(np.median(d)) if d.size else np.nan)
    if n < 6:
        return {**base, "p": np.nan, "method": "too few"}
    try:
        from scipy.stats import wilcoxon
        _, p = wilcoxon(top[m], deep[m]); method = "scipy"
    except Exception:
        rr = np.argsort(np.argsort(np.abs(nz))) + 1.0
        W = min(rr[nz > 0].sum(), rr[nz < 0].sum())
        mu = n * (n + 1) / 4.0; var = n * (n + 1) * (2 * n + 1) / 24.0
        z = (abs(W - mu) - 0.5) / _sqrt(var)
        p = min(max(2 * (1 - 0.5 * (1 + erf(z / _sqrt(2)))), 0.0), 1.0)
        method = "normal-approx"
    return {**base, "p": float(p), "method": method}


def _col(group, c):
    return pd.to_numeric(soak[soak["group"] == group][c], errors="coerce").dropna().to_numpy()

# near-surface metrics, each LOWER = cleaner, compared on the SAME variable across both cruises
NS_METRICS = [("sal_spike_top",  "near-surface salinity spiking",    "PSU"),
              ("temp_spike_top", "near-surface temperature spiking", "degC"),
              ("flag_pct_top",   "near-surface loop-edit flag %",    "%")]

def _esize(rb):
    m = abs(rb)
    return ("negligible" if m < 0.1 else "small" if m < 0.3 else "moderate" if m < 0.5 else "large")

between = {}
if not soak.empty:
    print("Between-group test (soak vs no-soak), same variable, two-sided Mann-Whitney:")
    print(f"{'metric':<34}{'soak med':>11}{'no-soak med':>13}{'p':>9}{'effect':>20}")
    print("-" * 85)
    for key, lab, unit in NS_METRICS:
        s, n = _col("soak", key), _col("no soak", key)
        if not len(s) or not len(n):
            continue
        r = mannwhitney(s, n)
        ms, mn = float(np.median(s)), float(np.median(n))
        es = _esize(r["rb"]) if np.isfinite(r["rb"]) else "n/a"
        between[key] = dict(lab=lab, unit=unit, soak_med=ms, nosoak_med=mn, esize=es, **r)
        eff = f"{es} ({r['rb']:+.2f})"
        print(f"{lab:<34}{ms:>11.4g}{mn:>13.4g}{r['p']:>9.3g}{eff:>20}")
    if between:
        print("method:", next(iter(between.values()))["method"])

# within-cast top-vs-deep (internally controlled: season/crew/sea state cancel within a cast)
within = {}
if not soak.empty:
    print("\nWithin-cast top-vs-deep salinity spiking (Wilcoxon; med_diff>0 = near-surface noisier than deep):")
    for g in ("soak", "no soak"):
        sub = soak[soak["group"] == g]
        w = wilcoxon_paired(pd.to_numeric(sub["sal_spike_top"], errors="coerce"),
                            pd.to_numeric(sub["sal_spike_deep"], errors="coerce"))
        st = np.nanmedian(pd.to_numeric(sub["sal_spike_top"], errors="coerce"))
        sd = np.nanmedian(pd.to_numeric(sub["sal_spike_deep"], errors="coerce"))
        ratio = (st / sd) if (sd and np.isfinite(sd) and sd > 0) else np.nan
        within[g] = dict(ratio=float(ratio), **w)
        print(f"  {g:<9} top/deep ratio {ratio:>5.2f}   med(top-deep) {w['med_diff']:+.5f}   p {w['p']:.3g}  (n={w['n']})")

## 8. Dose-response: soak time vs near-surface noise

If the soak is what helps, then *more* soak should give *cleaner* near-surface data, and that trend
should hold **within a single cruise** where season, sea state and crew are fixed. Soak duration is
measured per cast as the in-water time from first submersion (P >= 0.5 db) to the descent onset Section
uses, so it is exactly the window Section trims off the top of the cast.

- **Pooled (all casts):** screens the overall trend, but mixes the dose-response with the cross-cruise
  difference, so treat it as suggestive only.
- **Within cruise (soak only / no soak only):** the internally controlled test. A negative Spearman rho
  there (more soak, less noise) is the dose-response evidence least exposed to the cross-cruise confound.

Spearman (rank correlation, no linearity assumption), scipy or a Fisher-z numpy fallback so a p-value
always prints.

In [ ]:
from math import atanh as _atanh

def spearman(x, y):
    """Spearman rank correlation. scipy if present, else rank -> Pearson with a Fisher-z p-value, so a
    p always prints. rho < 0 = more soak goes with less near-surface noise (the hoped-for direction)."""
    x = np.asarray(x, float); y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y); x, y = x[m], y[m]
    n = len(x)
    if n < 4:
        return dict(rho=np.nan, p=np.nan, n=int(n), method="too few")
    try:
        from scipy.stats import spearmanr
        rho, p = spearmanr(x, y); method = "scipy"
    except Exception:
        def _rank(a):
            _, inv, cnt = np.unique(a, return_inverse=True, return_counts=True)
            o = a.argsort(); r = np.empty(len(a)); r[o] = np.arange(1, len(a) + 1)
            ssum = np.zeros(len(cnt)); np.add.at(ssum, inv, r); return (ssum / cnt)[inv]
        rx, ry = _rank(x), _rank(y)
        rho = float(np.corrcoef(rx, ry)[0, 1]) if np.std(rx) > 0 and np.std(ry) > 0 else np.nan
        if not np.isfinite(rho) or abs(rho) >= 1 or n <= 3:
            p = 0.0 if (np.isfinite(rho) and abs(rho) >= 1) else np.nan
        else:
            zf = _atanh(rho) * _sqrt(n - 3)
            p = min(max(2 * (1 - 0.5 * (1 + erf(abs(zf) / _sqrt(2)))), 0.0), 1.0)
        method = "numpy Fisher-z (pip install scipy for the exact test)"
    return dict(rho=float(rho), p=float(p), n=int(n), method=method)

DOSE = {}
if not soak.empty and "soak_s" in soak.columns and pd.to_numeric(soak["soak_s"], errors="coerce").notna().any():
    scopes = [("all casts", soak)]
    for g in ("soak", "no soak"):
        sub = soak[soak["group"] == g]
        if len(sub) >= 4:
            scopes.append((f"{g} only", sub))
    print("Dose-response: soak seconds vs near-surface metric (Spearman; rho<0 = more soak, cleaner)")
    print(f"\n  {'scope':<16}{'metric':<32}{'rho':>7}{'p':>10}{'n':>5}")
    print("  " + "-" * 70)
    for sclab, sdf in scopes:
        for key, lab, unit in NS_METRICS:
            r = spearman(pd.to_numeric(sdf["soak_s"], errors="coerce"),
                         pd.to_numeric(sdf[key], errors="coerce"))
            DOSE[(sclab, key)] = dict(lab=lab, **r)
            print(f"  {sclab:<16}{lab:<32}{r['rho']:>7.2f}{r['p']:>10.3g}{r['n']:>5}")
    _m = next((v["method"] for v in DOSE.values() if v.get("method", "").startswith(("scipy", "numpy"))), "n/a")
    print("\n  method:", _m)
    print("  median soak:  soak group %.0f s   |   no-soak group %.0f s"
          % (np.nanmedian(pd.to_numeric(soak[soak['group']=='soak']['soak_s'], errors='coerce')),
             np.nanmedian(pd.to_numeric(soak[soak['group']=='no soak']['soak_s'], errors='coerce'))))

    # scatter: soak seconds vs near-surface salinity spiking, coloured by cruise group
    fig, axg = new_static_fig(1, 1, figsize=(6.6, 5.0)); ax = axg[0][0]
    for g, col in (("soak", C_SOAK), ("no soak", C_NOSOAK)):
        sub = soak[soak["group"] == g]
        ax.scatter(pd.to_numeric(sub["soak_s"], errors="coerce"),
                   pd.to_numeric(sub["sal_spike_top"], errors="coerce"),
                   s=26, color=col, alpha=0.75, edgecolors="none", label=f"{g} (n={len(sub)})")
    rall = DOSE.get(("all casts", "sal_spike_top"), {})
    rsk  = DOSE.get(("soak only", "sal_spike_top"), {})
    ax.set_xlabel("surface soak (s)  [first submersion to descent onset]")
    ax.set_ylabel("near-surface salinity spiking  (std of S - running median, PSU)")
    ttl = "Dose-response: soak time vs near-surface salinity spiking"
    if rall: ttl += f"\npooled rho={rall.get('rho', float('nan')):.2f} (p={rall.get('p', float('nan')):.3g}, n={rall.get('n','?')})"
    if rsk and np.isfinite(rsk.get("rho", np.nan)):
        ttl += f"   |   soak-cruise rho={rsk['rho']:.2f} (p={rsk['p']:.3g}, n={rsk['n']})"
    ax.set_title(ttl, fontsize=9); ax.legend(frameon=False, fontsize=8); _despine(ax)
    fig.text(0.01, 0.005, "Pooled across cruises this mixes a dose-response with the cross-cruise "
             "difference; the soak-cruise-only rho isolates the within-cruise dose-response.",
             fontsize=7, color="#666666", va="bottom")
    fig.tight_layout(rect=[0, 0.05, 1, 1])
    finish(fig, OUT_DIR / "soak_dose_response.png")
else:
    print("soak_s not available (re-run section 5 after this update so process_cast records soak time).")

## 9. Automated inference

Reads the numbers computed above and writes the interpretation itself, in the observational register:
direction and relative size per metric, whether each is statistically distinguishable, the within-cast
contrast, then an overall verdict that states this as an **association** (not proof) and names the clean
next-step experiment. Saved to `soak_inference.txt` so it can be pasted straight into the handout or a
slide. The wording is generated from the data, so it stays correct if you re-run on other cruises.

In [ ]:
def _sentence_between(d):
    if d["soak_med"] < d["nosoak_med"]:   direction = "lower (cleaner)"
    elif d["soak_med"] > d["nosoak_med"]: direction = "higher (noisier)"
    else:                                 direction = "about equal"
    if np.isfinite(d["nosoak_med"]) and d["nosoak_med"] != 0:
        pct = 100 * (d["nosoak_med"] - d["soak_med"]) / d["nosoak_med"]
        relc = f", {abs(pct):.0f}% {'lower' if pct > 0 else 'higher'} than no-soak"
    else:
        relc = ""
    if not np.isfinite(d["p"]):
        sig = "not testable at these sample sizes"
    elif d["p"] < 0.05:
        sig = "statistically distinguishable"
    else:
        sig = "not statistically distinguishable at these sample sizes"
    return (f"- {d['lab'][0].upper()}{d['lab'][1:]}: soak median {d['soak_med']:.4g} {d['unit']} vs "
            f"no-soak {d['nosoak_med']:.4g} {d['unit']} ({direction}{relc}); "
            f"{sig} (p = {d['p']:.3g}, {d['esize']} effect).")

lines = ["AUTOMATED INFERENCE  -  near-surface data quality, soak vs no-soak",
         f"({', '.join(SOAK_CRUISES)} = soak;  {', '.join(NOSOAK_CRUISES)} = no soak;  "
         f"near-surface = top {TOP_DB:g} db of the sectioned downcast)", ""]

cleaner = distinct = n_tested = 0
for key, lab, unit in NS_METRICS:
    if key in between:
        d = between[key]; n_tested += 1
        lines.append(_sentence_between(d))
        if d["soak_med"] < d["nosoak_med"]:
            cleaner += 1
            if np.isfinite(d["p"]) and d["p"] < 0.05:
                distinct += 1
lines.append("")

if "soak" in within and "no soak" in within:
    rs, rn = within["soak"]["ratio"], within["no soak"]["ratio"]
    if np.isfinite(rs) and np.isfinite(rn):
        if rs < 1 <= rn:
            lines.append(
                f"- Within-cast control (internally controlled): in the soak casts the near-surface is "
                f"QUIETER than the same cast's deep water (top/deep salinity-spike ratio {rs:.2f}), while "
                f"in the no-soak casts it is NOISIER than deep ({rn:.2f}). Because each cast is compared to "
                f"itself, season, crew and sea state cancel, so this is the most defensible single result.")
        else:
            lines.append(
                f"- Within-cast control: top/deep salinity-spike ratio {rs:.2f} (soak) vs {rn:.2f} "
                f"(no soak). A ratio > 1 means the near-surface is noisier than that cast's own deep water.")
    lines.append("")

_dose = globals().get("DOSE", {})
_dsoak = _dose.get(("soak only", "sal_spike_top"))
_dall  = _dose.get(("all casts", "sal_spike_top"))
_d = (_dsoak if (_dsoak and np.isfinite(_dsoak.get("rho", np.nan)))
      else (_dall if (_dall and np.isfinite(_dall.get("rho", np.nan))) else None))
if _d is not None:
    if _d["rho"] < 0:   _sense = "more soak time goes with LESS near-surface salinity noise (cleaner)"
    elif _d["rho"] > 0: _sense = "more soak time goes with MORE near-surface salinity noise"
    else:               _sense = "no monotonic trend with soak time"
    _sig = ("distinguishable" if (np.isfinite(_d["p"]) and _d["p"] < 0.05)
            else "not distinguishable at this sample size")
    if _d is _dsoak:
        _scope = "within the soak cruise(s)"
        _extra = " This is within one cruise, so season, sea state and crew are not confounds here."
    else:
        _scope = "pooled across cruises"
        _extra = (" Pooled, this mixes the dose-response with the cross-cruise difference; read it "
                  "alongside the within-cruise row.")
    lines.append(f"- Dose-response ({_scope}): Spearman rho = {_d['rho']:+.2f} between soak seconds and "
                 f"near-surface salinity spiking (n = {_d['n']}); {_sense}; {_sig} (p = {_d['p']:.3g}).{_extra}")
    lines.append("")

if n_tested == 0:
    overall = "No near-surface metrics could be tested (check that both groups have casts)."
elif cleaner == n_tested:
    tail = (f", distinguishable on {distinct} of {n_tested}" if distinct
            else " (not all reach significance at these sample sizes)")
    overall = f"the soak group is cleaner on every near-surface metric tested{tail}."
elif cleaner > n_tested / 2:
    overall = f"the soak group is cleaner on {cleaner} of {n_tested} near-surface metrics ({distinct} distinguishable)."
else:
    overall = (f"the soak group is cleaner on only {cleaner} of {n_tested} near-surface metrics; "
               f"the data do not show a consistent near-surface advantage.")

lines.append("VERDICT (observational): " + overall[0].upper() + overall[1:])
lines.append(
    "This is an association across two cruises that also differ in season, sea state and crew, so it is "
    "CONSISTENT WITH the surface soak improving near-surface data quality but does not prove the soak "
    "alone caused it. Near-surface loop-edit flags are partly ship heave, which a steady descent (not the "
    "soak) addresses. The clean test is alternating soak / no-soak casts within one cruise under the same "
    "conditions: that is the recommended next step.")

text = "\n".join(lines)
print(text)
try:
    (OUT_DIR / "soak_inference.txt").write_text(text, encoding="utf-8")
    print("\nsaved  ->", OUT_DIR / "soak_inference.txt")
except Exception as e:
    print("save failed:", e)

## 10. Operational recommendation: soak time and cast ranges

Turns the analysis into a number the deck can act on, with the uncertainty kept visible:

1. **Proposed soak-time ranges.** Casts are grouped into `0-15 / 15-45 / 45-90 / >=90 s` and the median
   near-surface spiking is reported per range. This shows *where the improvement happens* rather than a
   single correlation.
2. **Recommended minimum soak.** The lowest-soak range whose near-surface spiking is within 15% of the
   best range (the plateau: the point beyond which waiting longer stops helping), reconciled with
   standard practice (Sea-Bird: a minimum soak of roughly 60 s for pump priming and TC-cell / thermistor
   equilibration; GO-SHIP: lower to ~10-20 m, hold ~1-2 min, return near-surface, then profile).
3. **Equilibration tail.** On the cleanest casts, the depth by which near-surface salinity spiking has
   fallen to the deep baseline, i.e. how much of the top of the cast is still settling even after a soak.

Saved to `soak_recommendation.txt`. **Provisional: from two cruises only. Confirm with an alternating
soak / no-soak sweep within one cruise before adopting as a standing rule.**

Sources: Sea-Bird Scientific SBE 19plus V2 operations / application notes; SBE Data Processing manual;
GO-SHIP / WHP CTD operations guidance (McTaggart et al.).

In [ ]:
# ---- proposed soak-time ranges + recommended minimum soak + equilibration tail ----
SOAK_BINS   = [0, 15, 45, 90, np.inf]
SOAK_LABELS = ["0-15 s", "15-45 s", "45-90 s", ">=90 s"]
TOL_PLATEAU = 1.15        # within 15% of the best range counts as "on the plateau"
MIN_BIN_N   = 3           # a range needs at least this many casts to anchor a recommendation
CLEAN_K     = 4           # cleanest soak casts used to estimate the equilibration tail

def _medc(d_, c):
    v = pd.to_numeric(d_[c], errors="coerce").dropna()
    return float(v.median()) if len(v) else np.nan

_PROFILE_CACHE = {}
def get_downcast(cruise, cast):
    """Re-process one cast and cache its near-surface downcast (P, S, T, descent rate). Used by the
    equilibration estimate and the reference-profile figure. Returns None if the cast cannot be built."""
    k = (cruise, cast)
    if k not in _PROFILE_CACHE:
        prof = None
        try:
            df = process_cast(cruise, cast)
        except Exception:
            df = None
        if df is not None:
            pcol = col_P(df); p = num(df, pcol) if pcol else None
            if p is not None and np.isfinite(p).any():
                imax = int(np.nanargmax(p)); dn = df.iloc[:imax + 1]
                rate = np.nan; tcol = col_time(dn)
                if tcol and "time" in tcol.lower():
                    t = num(dn, tcol)
                    if t is not None and np.isfinite(t).any():
                        dt = np.nanmax(t) - np.nanmin(t)
                        if dt > 0: rate = (np.nanmax(p) - np.nanmin(p)) / dt
                prof = dict(P=num(dn, col_P(dn)), S=salinity_of(dn), T=num(dn, col_T(dn)), rate=float(rate))
        _PROFILE_CACHE[k] = prof
    return _PROFILE_CACHE[k]

def settling_depth(prof, tol=1.2, win=7, max_db=30.0):
    """Shallowest depth (db) below which near-surface salinity spiking stays within tol x the cast's own
    deep baseline: an estimate of where equilibration has completed. Returns (depth_db, deep_spike)."""
    if not prof or prof.get("S") is None: return np.nan, np.nan
    P = np.asarray(prof["P"], float); S = np.asarray(prof["S"], float)
    m = np.isfinite(P) & np.isfinite(S); P, S = P[m], S[m]
    if P.size < 20: return np.nan, np.nan
    o = np.argsort(P); P, S = P[o], S[o]
    resid = S - pd.Series(S).rolling(win, center=True, min_periods=1).median().to_numpy()
    deepsel = P > TOP_DB
    deep = float(np.nanstd(resid[deepsel])) if deepsel.sum() > 5 else np.nan
    if not np.isfinite(deep) or deep <= 0: return np.nan, deep
    thr = tol * deep
    for d in np.arange(1.0, min(max_db, float(np.nanmax(P))), 1.0):
        w = (P >= d - 2.0) & (P <= d + 2.0)
        if w.sum() >= 5 and np.nanstd(resid[w]) <= thr:
            return float(d), deep
    return np.nan, deep

rec_lines = ["SOAK / EQUILIBRATION RECOMMENDATION  (data-driven, provisional)"]
_have = (not soak.empty and "soak_s" in soak.columns
         and pd.to_numeric(soak["soak_s"], errors="coerce").notna().sum() >= 4)
if _have:
    sk = soak.copy(); sk["soak_s"] = pd.to_numeric(sk["soak_s"], errors="coerce")
    sk["bin"] = pd.cut(sk["soak_s"], bins=SOAK_BINS, labels=SOAK_LABELS, right=False)
    print("Near-surface quality by proposed soak-time range (medians; lower = cleaner):")
    print(f"  {'range':<10}{'n':>4}{'sal spike':>12}{'temp spike':>13}{'flag %':>9}")
    print("  " + "-" * 48)
    st = {}
    for lab in SOAK_LABELS:
        gg = sk[sk["bin"] == lab]; n = len(gg)
        st[lab] = dict(n=n, sal=_medc(gg, "sal_spike_top"), temp=_medc(gg, "temp_spike_top"),
                       flag=_medc(gg, "flag_pct_top"), lo=SOAK_BINS[SOAK_LABELS.index(lab)])
        print(f"  {lab:<10}{n:>4}{st[lab]['sal']:>12.4g}{st[lab]['temp']:>13.4g}{st[lab]['flag']:>9.3g}")

    valid = {k: v for k, v in st.items() if v["n"] >= MIN_BIN_N and np.isfinite(v["sal"])}
    rec = None
    if valid:
        best = min(v["sal"] for v in valid.values())
        for lab in SOAK_LABELS:
            v = valid.get(lab)
            if v and v["sal"] <= TOL_PLATEAU * best:
                rec = v["lo"]; break

    # equilibration tail on the cleanest soak casts (re-processes a few casts)
    settle_ds, rates = [], []
    soakgrp = soak[soak["group"] == "soak"].dropna(subset=["sal_spike_top"]).sort_values("sal_spike_top").head(CLEAN_K)
    for _, r in soakgrp.iterrows():
        prof = get_downcast(r["cruise"], r["cast"])
        d, _dp = settling_depth(prof)
        if np.isfinite(d): settle_ds.append(d)
        if prof and np.isfinite(prof.get("rate", np.nan)): rates.append(prof["rate"])

    print()
    if rec is not None:
        rec_lines.append(f"- Data-driven minimum soak: near-surface salinity spiking stops improving "
                         f"(plateaus) by about {rec:g} s.")
        if rec < 60:
            rec_lines.append("- Standard practice: Sea-Bird advises a minimum soak of roughly 60 s (pump "
                             "priming, TC-cell and thermistor equilibration). The data plateau below that, "
                             "so the usual minimum already covers it.")
            proposed = 60
        else:
            rec_lines.append(f"- Standard practice: Sea-Bird advises a minimum soak of roughly 60 s; the data "
                             f"support that, with the plateau near {rec:g} s.")
            proposed = rec
        rec_lines.append(f"- PROPOSED SETTING: soak at least {proposed:g} s before starting the downcast "
                         f"(longer in rough seas, or when the top few metres matter most).")
    else:
        rec_lines.append("- Not enough spread in soak time to locate a plateau from these casts; the "
                         "soak-vs-no-soak contrast and the dose-response are the evidence. A dedicated soak "
                         "sweep is needed to set a specific threshold.")

    if settle_ds:
        md_ = float(np.median(settle_ds)); rate = float(np.nanmedian(rates)) if rates else np.nan
        tline = f" (~{md_ / rate:.0f} s of descent at the median {rate:.2f} db/s)" if (np.isfinite(rate) and rate > 0) else ""
        rec_lines.append(f"- Equilibration tail: even on the cleanest casts the top ~{md_:.0f} db still show "
                         f"elevated salinity spiking{tline}. Either extend the surface soak to absorb it, or "
                         f"treat the top ~{md_:.0f} db as provisional in the final product.")

    rec_lines.append("- CAVEAT: from two cruises only (soak n=%d). Provisional. Confirm with an alternating "
                     "soak / no-soak sweep within one cruise before adopting it as a standing rule."
                     % int((soak["group"] == "soak").sum()))
    txt = "\n".join(rec_lines); print(txt)
    try:
        (OUT_DIR / "soak_recommendation.txt").write_text(txt, encoding="utf-8")
        print("\nsaved  ->", OUT_DIR / "soak_recommendation.txt")
    except Exception as e:
        print("save failed:", e)
else:
    print("Need soak_s on >=4 casts; re-run section 5 after this update so process_cast records soak time.")

## 11. Reference profiles: what "clean" looks like

The cleanest and the noisiest near-surface profile in the fleet, same variable and same depth range, so
the deck can see the target. A clean near-surface trace is near-vertical with little wiggle in the top
few db; a noisy one shows a sawtooth or an overshoot where the conductivity cell and thermistor had not
equilibrated. This is the picture to aim for when setting the soak.

In [ ]:
# ---- cleanest vs noisiest near-surface profile ----
if not soak.empty and {"cruise", "cast", "sal_spike_top"}.issubset(soak.columns):
    ss = soak.dropna(subset=["sal_spike_top"])
    if len(ss) >= 2:
        clean = ss.loc[ss["sal_spike_top"].idxmin()]
        noisy = ss.loc[ss["sal_spike_top"].idxmax()]
        pc = get_downcast(clean["cruise"], clean["cast"])
        pn = get_downcast(noisy["cruise"], noisy["cast"])
        if pc and pn:
            VIEW_DB = 25.0
            fig, axg = new_static_fig(1, 2, figsize=(9.2, 5.6))
            for ax, var, lab, unit in ((axg[0][0], "S", "salinity", "PSU"),
                                       (axg[0][1], "T", "temperature", "degC")):
                for prof, name, meta, col in ((pc, "cleanest", clean, C_SOAK),
                                              (pn, "noisiest", noisy, C_NOSOAK)):
                    X = prof.get(var); P = prof.get("P")
                    if X is None or P is None: continue
                    X = np.asarray(X, float); P = np.asarray(P, float)
                    m = np.isfinite(P) & np.isfinite(X) & (P <= VIEW_DB)
                    sks = meta["soak_s"]; sstr = f"{sks:.0f} s" if np.isfinite(sks) else "soak n/a"
                    ax.plot(X[m], P[m], color=col, lw=1.1,
                            label=f"{name}: {meta['cruise']} {meta['cast']} ({sstr})")
                ax.invert_yaxis(); ax.set_xlabel(f"{lab} ({unit})"); _despine(ax)
                ax.set_title(f"Near-surface {lab}", fontsize=10)
            axg[0][0].set_ylabel("pressure (db)")
            axg[0][0].legend(frameon=False, fontsize=7, loc="lower right")
            fig.suptitle("What good looks like: cleanest vs noisiest near-surface profile (top %g db)" % VIEW_DB,
                         fontsize=11, y=0.98)
            fig.text(0.01, 0.005, "Clean = near-vertical, little wiggle in the top few db; noisy = sawtooth "
                     "/ overshoot where sensors had not equilibrated. Same variable, same depth range.",
                     fontsize=7, color="#666666", va="bottom")
            fig.tight_layout(rect=[0, 0.05, 1, 0.95])
            finish(fig, OUT_DIR / "soak_reference_profiles.png")
            print(f"cleanest: {clean['cruise']} {clean['cast']}  sal spike {clean['sal_spike_top']:.4g}  "
                  f"soak {clean['soak_s']:.0f} s" if np.isfinite(clean["soak_s"]) else
                  f"cleanest: {clean['cruise']} {clean['cast']}  sal spike {clean['sal_spike_top']:.4g}")
            print(f"noisiest: {noisy['cruise']} {noisy['cast']}  sal spike {noisy['sal_spike_top']:.4g}  "
                  f"soak {noisy['soak_s']:.0f} s" if np.isfinite(noisy["soak_s"]) else
                  f"noisiest: {noisy['cruise']} {noisy['cast']}  sal spike {noisy['sal_spike_top']:.4g}")
        else:
            print("Could not rebuild the cleanest/noisiest cast profiles (run on the processing machine).")
    else:
        print("Need >=2 casts with a near-surface salinity-spike value.")